### Destroyer

Select `Run All` to tear down this catalog's runtime resources and demo schemas.
The catalog itself is not dropped — create it in the workspace before deploy.


In [0]:
%pip install databricks-sdk --upgrade

In [0]:
%restart_python

In [ ]:
import sys
sys.path.append('./utils')

CATALOG = dbutils.widgets.get("CATALOG")
print(f"Destroying resources for catalog: {CATALOG}")

state = None
try:
    from uc_state import create_state_manager
    state = create_state_manager(CATALOG)
    print(f"State manager initialized for {CATALOG}")
except Exception as e:
    print(f"⚠️ Could not initialize state manager: {e}")
    print("This usually means the catalog or state table doesn't exist. Skipping resource cleanup.")

In [ ]:
if state:
    try:
        results = state.clear_all()
        for rtype, info in results.items():
            if info["successful"]:
                print(f"✅ {rtype}: deleted {len(info['successful'])} resources")
            if info["failed"]:
                print(f"⚠️ {rtype}: {len(info['failed'])} failures")
                for f in info["failed"]:
                    print(f"   - {f['name']}: {f['reason']}")
    except Exception as e:
        print(f"⚠️ clear_all() failed: {e}")
        print("Continuing with schema sweep (catalog is not dropped).")
else:
    print("No state manager available. Skipping tracked-resource cleanup; demo schemas still drop below. Catalog is not dropped.")

In [ ]:
# Live Genie Code functions in `{catalog}.ai` (e.g. active_crossing_status).
# The initializer also creates functions in this schema; those go away when
# the `ai` schema is dropped in the next cell. This list is only the name
# operators type during the talk, in case schema drop is skipped.
_DEMO_UC_FUNCTIONS = [
    "active_crossing_status",
]
_UC_SCHEMA = f"`{CATALOG}`.ai"


def _sql_gone(e):
    msg = str(e).lower()
    return any(s in msg for s in (
        "does not exist", "not found", "cannot find", "unknown function",
    ))


print(f"\n🔎 Sweep: demo-created UC functions in {_UC_SCHEMA}")
for _fn in _DEMO_UC_FUNCTIONS:
    _stmt = f"DROP FUNCTION IF EXISTS {_UC_SCHEMA}.{_fn}"
    try:
        spark.sql(_stmt)
        print(f"  ✅ dropped {_UC_SCHEMA}.{_fn}")
    except Exception as e:
        if _sql_gone(e):
            print(f"  ↪︎ already gone     {_UC_SCHEMA}.{_fn}")
        else:
            print(f"  ⚠️  {_UC_SCHEMA}.{_fn}: {type(e).__name__}: {e}")


In [0]:
# Demo schemas only — do not drop the pre-existing catalog.
_DEMO_SCHEMAS = [
    "metadata",
    "orders",
    "lakebase",
    "ai",
    "_internal_state",
    "_lakebase_storage",
    "_caspers_state",
]
print(f"\n🔍 Sweep: demo schemas in `{CATALOG}`")
for _schema in _DEMO_SCHEMAS:
    _fqn = f"`{CATALOG}`.`{_schema}`"
    try:
        spark.sql(f"DROP SCHEMA IF EXISTS {_fqn} CASCADE")
        print(f"  ✅ dropped {_fqn}")
    except Exception as e:
        print(f"  ⚠️  {_fqn}: {type(e).__name__}: {e}")
